# LightGBM -- train

In [ ]:

# =========================
# LightGBM 
# =========================
import lightgbm as lgb
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight



# =========================
# Custom metric
# =========================
def n_lgb_f1_score(y_pred, dataset):
    y_true = dataset.get_label()
    y_pred = y_pred.reshape(len(y_true), -1)
    y_pred_idx = np.argmax(y_pred, axis=1)
    score = f1_score(y_true, y_pred_idx, average='macro')
    return 'macro_f1', score, True


# =========================
# Class weights 
# =========================
def LightGBM_Training(X_trn, y_trn, X_val, y_val):
    

    classes = np.unique(y_trn)
    class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_trn
    )
    class_weight_dict = {i: w for i, w in enumerate(class_weights)}

    train_data = lgb.Dataset(X_trn, label=y_trn, weight=[class_weight_dict[y] for y in y_trn])
    val_data = lgb.Dataset(X_val_aug[:,-1,:], label=y_val, reference=train_data)


# =========================
# IMPROVED PARAMS
# =========================
    params = {
    'objective': 'multiclass',
    'num_class': 5,

    # learning
    'learning_rate': 0.05,

    # tree complexity (overfitting control)
    'num_leaves': 255,
    'max_depth': 8,
    'min_data_in_leaf': 50,

    # feature handling
    'feature_fraction': 0.7,
    'feature_fraction_bynode': 0.7,

    # bagging
    'bagging_fraction': 0.7,
    'bagging_freq': 5,

    # regularization 
    'lambda_l1': 1.0,
    'lambda_l2': 2.0,

    # noise robustness
    'min_gain_to_split': 0.01,

    'max_bin': 31,

    'seed': 42,
    'verbosity': -1
    }


# =========================
# Train
# =========================
    bst = lgb.train(
    params,
    train_data,
    num_boost_round=5000,
    valid_sets=[val_data],
    valid_names=['valid'],
    feval=n_lgb_f1_score,
    callbacks=[
        lgb.log_evaluation(period=100),
        lgb.early_stopping(stopping_rounds=100)
    ]
    )

# =========================
# Evaluation
# =========================
    final_preds_prob = bst.predict(X_val)
    final_preds = np.argmax(final_preds_prob, axis=1)

    print(f"\nFinal Validation Accuracy: {accuracy_score(y_val, final_preds):.4f}")
    print(f"Final Validation F1: {f1_score(y_val, final_preds, average='macro'):.4f}")
    
    return bst

    


In [ ]:
bst = LightGBM_Training(X_tr_aug[:,-1,:], y_flat_tr,   X_val_aug[:,-1,:], y_flat_val)

